# MILP Energy Lab

Fresh, offline modular gas investment and commitment analysis. Adapted from [Modular Expansion with Unit Commitment](https://github.com/PyPSA/PyPSA/blob/c838aa498557cc8e27a9d3ed10d45e35c4b0b442/docs/examples/modular-committable.ipynb), PyPSA contributors, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

Changes: HiGHS, bounded parameters, exact initially-off transitions, solar and shedding, independent feasibility checks, isolated execution and genuine AGILAB batch measurement. The supplied source notebook is preserved and is not executed. Investment is a horizon charge, not annual economics.

In [ ]:
import json
import math
import sys
from pathlib import Path

if "PROJECT_ROOT" in globals():
    sys.path.insert(0, str(Path(PROJECT_ROOT)))
from energy_core import default_settings, solve_scenario, cpu_limits, make_batch
from energy_runner import run_benchmark, run_batch

reference = solve_scenario(default_settings())
assert reference["status"] == "optimal", reference.get("message")
assert reference["residuals"]["verified"]
assert reference["modules"] == 30
assert reference["active_modules"] == [20, 30, 25, 4]
assert reference["dispatch"] == [4000, 6000, 5000, 800]
assert reference["startup"] == [20, 10, 0, 0]
hand_objective = 30 * 200 + sum([4000, 6000, 5000, 800]) + sum([20, 30, 25, 4])
assert hand_objective == 21879
assert math.isclose(reference["objective"], hand_objective, abs_tol=1e-6)
for t in range(4):
    assert abs(reference["dispatch"][t] - reference["demand"][t]) < 1e-5
    assert 0 <= reference["active_modules"][t] <= reference["modules"]
    assert 20 * reference["active_modules"][t] <= reference["dispatch"][t] + 1e-5
    assert reference["dispatch"][t] <= 200 * reference["active_modules"][t] + 1e-5
print({"modules": reference["modules"], "objective": reference["objective"], "verified": True})


## Physical shortage, no incumbent
Ten modules cannot supply the 6,000 MW peak. This case must report infeasible, with no made-up objective or schedule.

In [ ]:
infeasible = solve_scenario(default_settings() | {"max_modules": 10})
assert infeasible["status"] == "infeasible", infeasible.get("message")
assert infeasible["objective"] is None
assert infeasible["dispatch"] == []
assert not infeasible["residuals"]["verified"]
print({"capacity_limited_status": infeasible["status"], "objective": infeasible["objective"]})


## Real AGILAB batch: one versus two workers
The same four deterministic varied scenarios run through the unchanged AGILAB worker engine. Spawn/import overhead is included in full wall time. This is local batch throughput, not distributed computing or acceleration of one MILP. Different schedules can have the same optimal objective. No timings are cached.

In [ ]:
batch = make_batch(default_settings(), 4)
environment = cpu_limits()
if environment["effective_cpus"] >= 2:
    measurement = run_benchmark(batch, 2)
    assert measurement["comparison"]["matches"], measurement["comparison"]
    assert measurement["sequential"]["batch_sha256"] == measurement["parallel"]["batch_sha256"]
    assert len(measurement["parallel"]["rows"]) == 4
    summary = {"workers": 2, "matching_results": True,
               "speedup": measurement["speedup"],
               "sequential_wall_seconds": measurement["sequential"]["wall_seconds"],
               "parallel_wall_seconds": measurement["parallel"]["wall_seconds"]}
else:
    measurement = run_batch(batch, 1)
    assert len(measurement["rows"]) == 4
    assert all(row["result"]["residuals"]["verified"] for row in measurement["rows"])
    summary = {"workers": 1, "scaling": "Unavailable: only one effective CPU",
               "wall_seconds": measurement["wall_seconds"]}
output = {"results": {"default_objective": reference["objective"],
                       "installed_modules": reference["modules"],
                       "active_modules": reference["active_modules"],
                       "reference_check": reference["residuals"],
                       "capacity_limited_status": infeasible["status"],
                       "agilab_batch_measurement": summary,
                       "full_reference": reference,
                       "full_batch_measurement": measurement}}
Path("results.json").write_text(json.dumps(output, indent=2, allow_nan=False), encoding="utf-8")
print(summary)
